# 04 · RAG A: ejemplos recuperados (dev)

> **Ejecutar en Kaggle:** GPU T4 ×2, Internet activado, secreto `HF_TOKEN`. Después de `03_llm`.
> **Salidas:** `results/llm/ragA_dev_*.csv` y `results/rag/vecinos_dev.csv`.

Para cada noticia de dev se recuperan las de **train** más parecidas (embeddings multilingües `bge-m3`, trozos de 300 palabras con 50 de solape) y se usan como ejemplos few-shot.

- **8 vecinos equilibrados** (4 Fake + 4 Verdadera), con el prompt elegido en 03. La única diferencia con el few-shot fijo k=8 de 03 es qué ejemplos entran.
- **Referencia a batir:** few-shot fijo k=8 de 03 (AUC dev 0,988 con `gemma4-31b`).
- La noticia se excluye de su propia búsqueda y los ejemplos pasan por la misma limpieza.

**Limitación:** train y dev no tienen COVID (60 % de test), así que en test los vecinos aportarían estilo, no hechos. Test no se usa.

In [1]:
import os, sys, subprocess, shutil
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
REPO, BRANCH = "https://github.com/Rodrigo-Romay/Practica-TAPLN.git", "feature/RAG"
PAQUETES = ['transformers', 'accelerate', 'bitsandbytes', 'sentence-transformers']

if IN_KAGGLE:
    os.environ["HF_HOME"] = "/tmp/hf"   # más disco libre que /kaggle/working
    ROOT = Path("/kaggle/working/Practica-TAPLN")

    def secreto(nombre):
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(nombre)
        except Exception:
            return None

    if not ROOT.exists():
        gh = secreto("GH_TOKEN")   # solo hace falta si el repo es privado
        url = REPO.replace("https://", f"https://{gh}@") if gh else REPO
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(ROOT)], check=True)
    if PAQUETES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PAQUETES], check=True)
    hf = secreto("HF_TOKEN")
    if hf:
        os.environ["HF_TOKEN"] = hf
    else:
        print("Sin HF_TOKEN: solo funcionarán los modelos no restringidos.")
    # Resultados intermedios no versionados: dataset privado `fakedes-resultados`
    extra = Path("/kaggle/input/fakedes-resultados/results")
    if extra.exists():
        shutil.copytree(extra, ROOT / "results", dirs_exist_ok=True)
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "data" / "processed" / "train.csv").exists():
    raw = ROOT / "data" / "raw"
    subprocess.run([sys.executable, str(ROOT / "src" / "process_data.py"),
                    "--train", str(raw / "train.xlsx"), "--dev", str(raw / "development.xlsx"),
                    "--test", str(raw / "test.xlsx"), "--out", str(ROOT / "data" / "processed")], check=True)

Sin HF_TOKEN: solo funcionarán los modelos no restringidos.
Duplicados train/dev eliminados de dev: 4
train: 676 filas | Fake=338 True=338 | titulares vacíos=0 | duplicados internos=0
dev: 291 filas | Fake=140 True=151 | titulares vacíos=0 | duplicados internos=1
test: 572 filas | Fake=286 True=286 | titulares vacíos=72 | duplicados internos=0
Guardado en /kaggle/working/Practica-TAPLN/data/processed


In [2]:
import numpy as np
import pandas as pd
import torch
import data_utils as du
import preprocessing as pp
import evaluation as ev
import experiments as ex
import llm
import llm_fewshot as fs

splits = du.load_clean_splits(ROOT)
media_re = pp.media_regex_from_splits(splits)   # nombres de medio de train/dev
HAY_GPU = torch.cuda.is_available()
MAX_TOKENS = 4500   # si un prompt no cabe, se quitan ejemplos; la noticia no se recorta 
print({k: len(v) for k, v in splits.items()}, "| GPU:", HAY_GPU)

{'train': 676, 'dev': 290, 'test': 572} | GPU: True


## Índice de ejemplos (train)

In [ ]:
from rag.index import ExampleIndex, STEmbedder, TfidfEmbedder, EMBEDDER

RAG = ROOT / "results" / "rag"
RAG.mkdir(parents=True, exist_ok=True)
pool = ex.pool_for(splits, media_re, "dev")          
ruta_indice = RAG / "indice_bge-m3_train.npz"
try:
    emb = STEmbedder(EMBEDDER)                     
    ix = ExampleIndex.load(ruta_indice, emb) if ruta_indice.exists() else ExampleIndex(emb).build(pool)
    ix.save(ruta_indice)
except ImportError:
    print("Sin sentence-transformers: se usa TF-IDF como sustituto (solo para probar el notebook en local).")
    ix = ExampleIndex(TfidfEmbedder()).build(pool)
print(f"{len(ix.docs)} noticias, {len(ix.chunks)} trozos, embedder: {ix.embedder.name}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

676 noticias, 1110 trozos, embedder: BAAI/bge-m3


### Calidad de la recuperación

¿Los vecinos comparten etiqueta con la noticia? Voto de los 5 vecinos de train (descriptivo, no es un sistema).

In [4]:
dev_in = ex.inputs_for(splits, media_re, "dev")
filas = []
for r in dev_in.itertuples(index=False):
    v = ix.search(r.body, 5, exclude=[f"dev:{r.Id}"], splits=("train",))
    filas.append({"Id": r.Id, "label": r.label, "frac_fake_vecinos": v["label"].mean(), "sim_max": v["sim"].max(),
                  "vecinos": "|".join(v["uid"])})
vec = pd.DataFrame(filas)
vec.to_csv(RAG / "vecinos_dev.csv", index=False)
print(ev.evaluate(vec.label, ev.predict_threshold(vec.frac_fake_vecinos, 0.5), vec.frac_fake_vecinos, "voto 5 vecinos"))
vec.sim_max.describe().round(3)

{'modelo': 'voto 5 vecinos', 'f1_fake': 0.519298245614035, 'precision': 0.5068493150684932, 'recall': 0.5323741007194245, 'pct_pred_fake': 0.503448275862069, 'pct_real_fake': 0.4793103448275862, 'n': 290, 'auc': np.float64(0.555433798656439)}


count    290.000
mean       0.635
std        0.081
min        0.460
25%        0.573
50%        0.643
75%        0.686
max        0.998
Name: sim_max, dtype: float64

## Experimentos

Los vecinos se calculan al crear el experimento; después se libera el embedder para dejar la GPU al LLM.

In [5]:
# Referencia a batir: la configuración elegida en 03 para cada modelo (few-shot fijo, k=8)
PROMPT = {"gemma4-31b": "base", "qwen38-27b": "detallado"}
REF = {"gemma4-31b": "ajuste_dev_gemma4-31b__base_k8.csv", "qwen38-27b": "ajuste_dev_qwen38-27b__detallado_k8.csv"}
# Mismo k y mismo prompt que en 03: lo único que cambia es qué ejemplos entran 
EXPS = {clave: [ex.experiment(splits, media_re, "dev", modo="rag_a", k=8, prompt=PROMPT[clave], index=ix,
                              per_class=True, max_tokens=MAX_TOKENS)]
        for clave in ex.FINALISTAS}
for clave, es in EXPS.items():
    for e in es:
        print(clave, e["experimento"], e["pid"])
print(ex.show_prompt(EXPS["gemma4-31b"][0]))
if hasattr(ix.embedder, "model"):
    ix.embedder.model = None      
llm.release()


gemma4-31b ragA ra8base-e26318
qwen38-27b ragA ra8detallado-75173c
[system] Eres un verificador de noticias en español. Lees una noticia y decides si es verdadera o falsa. En el texto se han eliminado enlaces y nombres de medios, y las cifras aparecen como NUM. Responde con una sola palabra: Verdadera o Falsa.

[user] <titular>
La masacre de Ayotzinapa, tres años sin respuestas
</titular>

<cuerpo_noticia>
La desaparición de los NUM estudiantes en México cumple NUM meses sin que el Gobierno haya sido capaz de desenredar el caso Este martes NUM de septiembre México ha amanecido con una acumulación de tragedias sin resolver. Una semana después de que un terremoto de magnitud NUM cimbrara el centro del p [...]
</cuerpo_noticia>

¿Esta noticia es Verdadera o Falsa? (Responde con una sola palabra).

[assistant] Verdadera

[user] <titular>
Responde Carmen Salinas a críticas: advierte que llevará hamaca a San Lázaro
</titular>

<cuerpo_noticia>
Carmen Salinas aseguró que está pensando en llev

## Ejecución (GPU)

In [6]:
MODELOS = list(ex.FINALISTAS)   # gemma4-31b (principal) y qwen38-27b (contraste)
if HAY_GPU:
    for clave in MODELOS:
        ex.run_all(ROOT, EXPS[clave], [clave], borrar_cache=IN_KAGGLE)
else:
    print("Sin GPU: no se ejecutan modelos. Con los CSV ya descargados en results/llm/ se puede ver el resumen.")

== gemma4-31b (google/gemma-4-31B-it): 1 experimento(s)


config.json:   0%|          | 0.00/4.62k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

... (16 outputs left out)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


-- ragA ra8detallado-75173c


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


  50/290  (13.70 s/noticia)
  100/290  (13.69 s/noticia)
  150/290  (13.62 s/noticia)
  200/290  (13.70 s/noticia)
  250/290  (13.65 s/noticia)
   {'n': 290, 'tokens_media': 2997.8172413793104, 'tokens_total': 869367, 's_noticia': 13.57720877302104, 'minutos_total': 65.62317573626837}


## Resultados en dev

In [7]:
COLS = ["modelo", "decision", "f1_fake", "f1_fake_inf", "f1_fake_sup", "auc", "auc_inf", "auc_sup",
        "precision", "recall", "pct_pred_fake", "s_noticia", "tokens", "completo"]
tabla = ex.dev_table(ROOT, "dev")
if len(tabla):
    tabla = tabla.sort_values("auc", ascending=False)
    tabla.to_csv(ROOT / "results" / "llm" / "ragA_dev_resumen.csv", index=False)
    display(tabla[COLS].round(3).reset_index(drop=True))

,modelo,decision,f1_fake,f1_fake_inf,f1_fake_sup,auc,auc_inf,auc_sup,precision,recall,pct_pred_fake,s_noticia,tokens,completo
0,ajuste_dev_gemma4-31b__base_k8,rango50 (transductiva),0.930,0.897,0.959,0.988,0.977,0.996,0.910,0.950,0.500,12.025,2712.741,290
1,ajuste_dev_gemma4-31b__base_k8,umbral,0.942,0.911,0.969,0.988,0.977,0.996,0.942,0.942,0.479,12.025,2712.741,290
2,ajuste_dev_qwen38-27b__detallado_k8,rango50 (transductiva),0.937,0.905,0.964,0.987,0.975,0.996,0.917,0.957,0.500,12.605,2961.052,290
3,ajuste_dev_qwen38-27b__detallado_k8,umbral,0.918,0.883,0.948,0.987,0.975,0.996,0.876,0.964,0.528,12.605,2961.052,290
4,ragA_dev_gemma4-31b_ra8base-e26318,rango50 (transductiva),0.951,0.924,0.974,0.986,0.971,0.997,0.931,0.971,0.500,13.257,2744.590,290
5,ragA_dev_gemma4-31b_ra8base-e26318,umbral,0.956,0.928,0.979,0.986,0.971,0.997,0.985,0.928,0.452,13.257,2744.590,290
6,ragA_dev_qwen38-27b_ra8detallado-75173c,rango50 (transductiva),0.944,0.913,0.969,0.985,0.970,0.997,0.924,0.964,0.500,13.577,2997.817,290
7,ragA_dev_qwen38-27b_ra8detallado-75173c,umbral,0.940,0.908,0.966,0.985,0.970,0.997,0.918,0.964,0.503,13.577,2997.817,290
8,ajuste_dev_gemma4-31b__base_k4,rango50 (transductiva),0.923,0.887,0.953,0.985,0.973,0.994,0.903,0.942,0.500,7.239,1593.741,290
9,ajuste_dev_gemma4-31b__base_k4,umbral,0.952,0.924,0.976,0.985,0.973,0.994,0.970,0.935,0.462,7.239,1593.741,290


In [8]:
# Comparación pareada de cada experimento nuevo frente a la configuración elegida en 03
# (few-shot fijo k=8) y frente al zero-shot del piloto, por modelo
filas = []
for clave in MODELOS:
    sistemas = {}
    for nombre, f in [("few-shot k=8 (03)", REF[clave]), ("zero-shot (piloto)", f"piloto_dev_{clave}.csv")]:
        if (ROOT / "results" / "llm" / f).exists():
            sistemas[nombre] = ex.to_predictions(pd.read_csv(ROOT / "results" / "llm" / f), nombre)
    for e in EXPS[clave]:
        p = ex.out_path(ROOT, e, clave)
        if fs.is_complete(p, len(e["inputs"])):
            sistemas[f"{e['experimento']} {e['pid']}"] = ex.to_predictions(pd.read_csv(p), e["pid"])
    nuevos = [s for s in sistemas if s not in ("few-shot k=8 (03)", "zero-shot (piloto)")]
    if nuevos and len(sistemas) > 1:
        c = ev.compare_predictions(ROOT, sistemas, "dev")
        filas.append(c[c["A"].isin(nuevos) ^ c["B"].isin(nuevos)].assign(modelo=clave))
if filas:
    display(pd.concat(filas, ignore_index=True).round(3))


,A,B,split,decision,dif_f1_fake,dif_f1_fake_inf,dif_f1_fake_sup,p_f1_fake,dif_auc,dif_auc_inf,dif_auc_sup,p_auc,modelo
0,few-shot k=8 (03),ragA ra8base-e26318,dev,umbral,-0.013,-0.037,0.008,0.264,0.002,-0.004,0.009,0.581,gemma4-31b
1,few-shot k=8 (03),ragA ra8base-e26318,dev,rango50 (transductiva),-0.021,-0.048,0.003,0.088,0.002,-0.004,0.009,0.581,gemma4-31b
2,zero-shot (piloto),ragA ra8base-e26318,dev,umbral,-0.042,-0.072,-0.014,0.005,-0.005,-0.013,0.002,0.208,gemma4-31b
3,zero-shot (piloto),ragA ra8base-e26318,dev,rango50 (transductiva),-0.021,-0.048,0.003,0.108,-0.005,-0.013,0.002,0.208,gemma4-31b
4,few-shot k=8 (03),ragA ra8detallado-75173c,dev,umbral,-0.023,-0.048,0.003,0.112,0.002,-0.004,0.008,0.658,qwen38-27b
5,few-shot k=8 (03),ragA ra8detallado-75173c,dev,rango50 (transductiva),-0.007,-0.030,0.015,0.549,0.002,-0.004,0.008,0.658,qwen38-27b
6,zero-shot (piloto),ragA ra8detallado-75173c,dev,umbral,-0.007,-0.039,0.026,0.660,-0.013,-0.025,-0.003,0.010,qwen38-27b
7,zero-shot (piloto),ragA ra8detallado-75173c,dev,rango50 (transductiva),-0.035,-0.065,-0.008,0.009,-0.013,-0.025,-0.003,0.010,qwen38-27b


In [9]:
# Desglose del mejor candidato de cada modelo: Fake satíricas / otras (dev) y COVID / no COVID
for clave in MODELOS:
    for e in EXPS[clave]:
        p = ex.out_path(ROOT, e, clave)
        if fs.is_complete(p, len(e["inputs"])):
            r = ev.evaluate_predictions(ROOT, ex.to_predictions(pd.read_csv(p), p.stem), "dev", p.stem, n_boot=200)
            d = r["desglose"]
            print(p.stem)
            display(d[d.decision == "umbral"][["desglose", "grupo", "n", "f1_fake", "auc", "pct_pred_fake"]].round(3))

ragA_dev_gemma4-31b_ra8base-e26318


,desglose,grupo,n,f1_fake,auc,pct_pred_fake
0,covid,COVID,12,1.000,1.000,0.417
1,covid,no COVID,278,0.954,0.985,0.453
2,longitud,401-800,79,0.966,0.967,0.354
3,longitud,<=400,191,0.957,0.991,0.534
4,longitud,>800,20,0.667,1.000,0.050
5,titular,con titular,290,0.956,0.986,0.452
6,satira,Fake satíricas + True,259,0.986,1.000,0.421
7,satira,Fake no satíricas + True,182,0.800,0.938,0.132


ragA_dev_qwen38-27b_ra8detallado-75173c


,desglose,grupo,n,f1_fake,auc,pct_pred_fake
0,covid,COVID,12,1.000,1.000,0.417
1,covid,no COVID,278,0.938,0.984,0.507
2,longitud,401-800,79,0.918,0.967,0.392
3,longitud,<=400,191,0.945,0.989,0.592
4,longitud,>800,20,1.000,1.000,0.100
5,titular,con titular,290,0.940,0.985,0.503
6,satira,Fake satíricas + True,259,0.947,1.000,0.463
7,satira,Fake no satíricas + True,182,0.754,0.935,0.209


## Lectura de resultados y decisión

- **Frente al few-shot fijo k=8 (03):** sin mejora significativa. AUC 0,986 frente a 0,988 en `gemma4-31b` (p = 0,58) y 0,985 frente a 0,987 en `qwen38-27b` (p = 0,66). El F1 con umbral sube algo en gemma (0,956 frente a 0,942) pero tampoco es significativo (p = 0,26).
- **Frente al zero-shot** sí mejora el F1 (gemma +0,042, qwen +0,035), pero la ganancia viene de dar ejemplos, no de que sean vecinos.
- **La recuperación aporta poca señal:** el voto de los 5 vecinos da AUC 0,555; los vecinos comparten etiqueta con la noticia casi al azar.
- **Fake no satíricas** (lo más parecido a test en dev): sube el F1 (gemma 0,762 → 0,800) pero baja el AUC (0,949 → 0,938). La subida se debe a dónde cae el umbral, no a separar mejor. Solo hay ~31 Fake, así que no es concluyente.
- **Coste:** ~13,3 s/noticia y ~2.750 tokens (gemma), frente a 12,0 s y 2.713 con k=8 fijo; ~65 min por modelo, más el índice y el embedder en GPU.

**Decisión:** no se adopta RAG A. Se mantiene el few-shot fijo k=8. Test no se usa.